# 2. EDA with PySpark

#### a) Show a histogram of numbers of reviews for each accomodation.

In [0]:
from pyspark.sql import functions as sf

df_eda = spark.read.table("airbnb.hosts.raw_supply_chain")

In [0]:
import matplotlib.pyplot as plt
reviews_df = df_eda.select("`number of reviews`").filter("`number of reviews` IS NOT NULL").toPandas()
reviews_series = reviews_df["number of reviews"]


plt.figure(figsize=(12, 6))
plt.hist(reviews_series, bins=50, color='red', edgecolor='black')

plt.title("Distribution of Number of Reviews", fontsize=16)
plt.xlabel("Number of Reviews", fontsize=12)
plt.ylabel("Frequency (Count)", fontsize=12)

plt.xticks(rotation=0) 
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

#### b) Clean the price column. Is it numeric?

In [0]:
df_price = df_eda.withColumn(
    "price_cleaned", 
    sf.trim(
        sf.regexp_replace(
            sf.regexp_replace("price", r"\$", ""), 
            r",", ""
        )
    )
)


#### c) Calculate the average price of accomodations per neighborhood group. In which neighborhood group is the average price highest?

In [0]:
df_clean = df_price.withColumn(
    "price", 
    sf.expr("try_cast(price_cleaned AS double)")
)
df_result = df_clean.groupBy("neighbourhood group") \
                      .agg(sf.avg("price").alias("avg_price")) \
                      .orderBy(sf.col("avg_price").desc())

df_result.show()

#### d) Show the top 10 accomodations with most reviews.

In [0]:
from pyspark.sql import functions as sf

df_clean = df_eda.dropna(subset=["number of reviews", "neighbourhood group"])

df_clean = df_clean.withColumn(
    "reviews_text_clean",
    sf.regexp_replace("number of reviews", r"\d{1,2}/\d{1,2}/\d{4}", "")
)

df_clean = df_clean.withColumn(
    "number of reviews_clean",
    sf.expr("try_cast(reviews_text_clean AS int)")
)

df_clean = df_clean.withColumn(
    "neighbourhood_group_clean",
    sf.trim(sf.regexp_replace("neighbourhood group", r"\d+", ""))
)

df_review = df_clean.groupBy("neighbourhood_group_clean") \
                    .agg(sf.max("number of reviews_clean").alias("most_reviews")) \
                    .orderBy(sf.col("most_reviews").desc())

df_review.show(10)

#### e) Show the top 10 accomodations with best reviews.

In [0]:
from pyspark.sql.functions import col, expr

df_clean = df_eda.dropna(subset=(["NAME", "review rate number"]))

df_review = (
    df_clean.select("NAME", "review rate number", "number of reviews")
    .filter(col("NAME").rlike("[a-zA-Z]"))
    .withColumn("review rate number", expr("try_cast(`review rate number` AS DOUBLE)"))
    .dropDuplicates(["NAME"])
    .withColumn("number of reviews", col("number of reviews").cast("integer"))
    .filter(col("review rate number").isNotNull())
    .orderBy(col("review rate number").desc(), col("number of reviews").desc())
    .show(10, truncate=False)
)

#### f) Other EDAs of your choice

#### Vilket neighborhood group har högst andel omedelbart bokningsbara boenden (instant_bookable)?

In [0]:
df_instant = df_eda.select("neighbourhood group", "instant_bookable") \
    .dropna(subset=["neighbourhood group", "instant_bookable"]) \
    .filter(sf.col("instant_bookable") == 'TRUE') \
    .filter(sf.col("neighbourhood group") != "brookln") \
    .filter(sf.col("neighbourhood group").isin(["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"])) \
    .groupBy("neighbourhood group") \
    .count() \
    .orderBy(sf.col("count").desc()) \
    .show(10)

In [0]:
df_eda.select("neighbourhood group") \
    .distinct() \
    .show()

#### Vilket rum-typ (room type) har högst genomsnittspris?

In [0]:
from pyspark.sql.functions import col, expr, avg

df = (
    df_eda.select("room type", "price")
    .withColumn("price", sf.regexp_replace("price", "\\$", ""))
    .withColumn("price", sf.regexp_replace("price", ",", ""))
    .withColumn("price", col("price").cast("double"))
    .dropna(subset=["room type", "price"])
    .filter(sf.col("room type").isin(["Entire home/apt", "Private room", "Shared room", "Hotel room"]))
    .groupBy("room type")
    .agg(avg("price").alias("avg_price"))
    .orderBy("avg_price")
    .show(10)
)